![iceberg-logo](https://www.apache.org/logos/res/iceberg/iceberg.png)

### [Docker, Spark, and Iceberg: The Fastest Way to Try Iceberg!](https://tabular.io/blog/docker-spark-and-iceberg/)

### SQL Authoring Note (Kotlin Kernel)

Known limitation (Kotlin kernel): JupyterLab does not apply SQL syntax highlighting inside Kotlin triple-quoted strings (`"""..."""`).

What works in this notebook:
- Use the toolbar **Format SQL** action for query formatting.
- Keep SQL in a dedicated variable and execute with `spark.sql(query)`.

```kotlin
val query = """
SELECT *
FROM nyc.taxis
""".trimIndent()

spark.sql(query).show(100, 0, false)
```


In [ ]:
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions.col

val spark = SparkSession.builder().appName("Jupyter").getOrCreate()

spark.version()

# Write support

This notebook demonstrates writing to Iceberg tables using Kotlin + Spark SQL. First, connect to the [catalog](https://iceberg.apache.org/concepts/catalog/#iceberg-catalogs), the place where tables are being tracked.

In [ ]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS default")

# Loading data using Arrow

Spark DataFrame is used to load a Parquet file into memory, and using Kotlin + Spark SQL this data can be written to an Iceberg table.

In [ ]:
val df = spark.read().parquet("/home/iceberg/data/yellow_tripdata_2022-01.parquet")

df

# Create an Iceberg table

Next create the Iceberg table directly from the `Spark DataFrame`.

In [ ]:
val table_name = "default.taxi_dataset"

spark.sql("DROP TABLE IF EXISTS $table_name")
df.limit(0).write().format("iceberg").mode("overwrite").saveAsTable(table_name)

spark.table(table_name)

# Write the data

Let's append the data to the table. Appending or overwriting is equivalent since the table is empty. Next we can query the table and see that the data is there.

In [ ]:
df.writeTo(table_name).append()

check(spark.table(table_name).count() == df.count())

spark.table(table_name)

In [ ]:
spark.sql("""
SELECT snapshot_id, committed_at, operation
FROM $table_name.snapshots
ORDER BY committed_at DESC
""".trimIndent()).show(false)

# Append data

Let's append another month of data to the table

In [ ]:
val df_feb = spark.read().parquet("/home/iceberg/data/yellow_tripdata_2022-02.parquet")
df_feb.writeTo(table_name).append()

In [ ]:
spark.sql("""
SELECT snapshot_id, committed_at, operation
FROM $table_name.snapshots
ORDER BY committed_at DESC
""".trimIndent()).show(false)

# Feature generation

Consider that we want to train a model to determine which features contribute to the tip amount. `tip_per_mile` is a good target to train the model on. When we try to append the data, we need to evolve the schema first.

In [ ]:
val enriched_df = spark.table(table_name)
    .withColumn("tip_per_mile", col("tip_amount") / col("trip_distance"))

try {
    enriched_df.writeTo(table_name).overwritePartitions()
} catch (e: Exception) {
    println("Error: ${e.message}")
}

In [ ]:
spark.sql("ALTER TABLE $table_name ADD COLUMN tip_per_mile DOUBLE")
spark.table(table_name).printSchema()

In [ ]:
enriched_df.writeTo(table_name).overwritePartitions()

spark.table(table_name)